In [16]:
import os
import json
import yaml
import numpy as np
import librosa
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import IPython
import soundfile as sf
from tqdm import tqdm 
from collections import OrderedDict
from sklearn.model_selection import train_test_split
import glob
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint
import tensorflow as tf

In [2]:
BASE_DIR = "."  


MEDLEYDB_ROOT = os.path.join(BASE_DIR, "MedleyDB")
AUDIO_DIR = os.path.join(MEDLEYDB_ROOT, "Audio")
ANNOTATIONS_DIR = os.path.join(MEDLEYDB_ROOT, "Annotations")


INSTRUMENT_ACTIVATIONS_DIR = os.path.join(ANNOTATIONS_DIR, "Instrument_Activations")
ACTIVATION_CONF_DIR = os.path.join(INSTRUMENT_ACTIVATIONS_DIR, "ACTIVATION_CONF")
SOURCEID_DIR = os.path.join(INSTRUMENT_ACTIVATIONS_DIR, "SOURCEID")

In [3]:
# First, we define the mapping we already agreed on
# Mapping from original instruments to grouped categories
instrument_mapping = {
    # Large individual classes
    'electric_bass': 'electric_bass',
    'drum_set': 'drum_set',
    'piano': 'piano',
    'tack_piano': 'piano',
    'synthesizer': 'synthesizer',
    'electric_piano': 'synthesizer',
    'clean_electric_guitar': 'clean_electric_guitar',
    'distorted_electric_guitar': 'distorted_electric_guitar',
    'acoustic_guitar': 'acoustic_guitar',
    'violin': 'violin',
    'female_singer': 'female_singer',
    'vocalists': 'vocalists',
    
    # Male vocals
    'male_singer': 'male_vocals',
    'male_speaker': 'male_vocals',
    'male_rapper': 'male_vocals',
    
    # Other bowed strings
    'viola': 'other_bowed_strings',
    'cello': 'other_bowed_strings',
    'double_bass': 'other_bowed_strings',
    'violin_section': 'other_bowed_strings',
    'viola_section': 'other_bowed_strings',
    'cello_section': 'other_bowed_strings',
    'string_section': 'other_bowed_strings',
    'erhu': 'other_bowed_strings',
    
    # Woodwinds
    'flute': 'woodwinds',
    'clarinet': 'woodwinds',
    'oboe': 'woodwinds',
    'bassoon': 'woodwinds',
    'tenor_saxophone': 'woodwinds',
    'soprano_saxophone': 'woodwinds',
    'alto_saxophone': 'woodwinds',
    'baritone_saxophone': 'woodwinds',
    'bass_clarinet': 'woodwinds',
    'piccolo': 'woodwinds',
    'flute_section': 'woodwinds',
    'clarinet_section': 'woodwinds',
    
    # Brass
    'trumpet': 'brass',
    'trombone': 'brass',
    'french_horn': 'brass',
    'french_horn_section': 'brass',
    'brass_section': 'brass',
    'tuba': 'brass',
    'horn_section': 'brass',
    'trumpet_section': 'brass',
    
    # Other percussion
    'auxiliary_percussion': 'other_percussion',
    'tambourine': 'other_percussion',
    'cymbals': 'other_percussion',
    'timpani': 'other_percussion',
    'tabla': 'other_percussion',
    'shaker': 'other_percussion',
    'snare_drum': 'other_percussion',
    'bass_drum': 'other_percussion',
    'kick_drum': 'other_percussion',
    'bongo': 'other_percussion',
    'darbuka': 'other_percussion',
    'doumbek': 'other_percussion',
    'gong': 'other_percussion',
    'toms': 'other_percussion',
    'drum_machine': 'other_percussion',
    'claps': 'other_percussion',
    'chimes': 'other_percussion',
    
    # Other plucked strings
    'mandolin': 'other_plucked_strings',
    'banjo': 'other_plucked_strings',
    'guzheng': 'other_plucked_strings',
    'oud': 'other_plucked_strings',
    'harp': 'other_plucked_strings',
    'lap_steel_guitar': 'other_plucked_strings',
    'yangqin': 'other_plucked_strings',
    'zhongruan': 'other_plucked_strings',
    'glockenspiel': 'other_plucked_strings',
    'vibraphone': 'other_plucked_strings',
    
    # Reed instruments
    'accordion': 'reed_instruments',
    'harmonica': 'reed_instruments',
    'melodica': 'reed_instruments',
    
    # Effects and processed sounds
    'fx_slash_processed_sound': 'fx_and_processed',
    'scratches': 'fx_and_processed'
}

# List of instruments to completely discard
instruments_to_discard = [
    'sampler',
    'main_system',
    'trombone_section',
]

# Final categories
instrument_categories = [
    'electric_bass',
    'drum_set',
    'piano',
    'synthesizer',
    'clean_electric_guitar',
    'distorted_electric_guitar',
    'acoustic_guitar',
    'violin',
    'female_singer',
    'vocalists',
    'male_vocals',
    'other_bowed_strings',
    'woodwinds',
    'brass',
    'other_percussion',
    'other_plucked_strings',
    'reed_instruments',
    'fx_and_processed'
]

# Mapping from categories to indices
category_to_idx = {category: idx for idx, category in enumerate(instrument_categories)}

In [14]:
# 1️⃣ Load the features
mel_features = np.load("extracted_mix_features/all_mels.npy", allow_pickle=True)
cqt_features = np.load("extracted_mix_features/all_cqts.npy", allow_pickle=True)
labels = np.load("extracted_mix_features/all_labels.npy", allow_pickle=True)



# 🟡 Conversione in float32 e rimozione dimensioni inutili
X_mel = np.stack([np.squeeze(f).astype(np.float32) for f in mel_features])  # (batch, 96, 259)
X_cqt = np.stack([np.squeeze(f).astype(np.float32) for f in cqt_features])  # (batch, 96, 259)
y = np.stack([l.astype(np.float32) for l in labels])                        # (batch, num_labels)


In [18]:

# 🟡 Normalizzazione min-max
X_mel = (X_mel - np.min(X_mel)) / (np.max(X_mel) - np.min(X_mel))
X_cqt = (X_cqt - np.min(X_cqt)) / (np.max(X_cqt) - np.min(X_cqt))

# 🟡 Stacka mel e cqt lungo il canale finale
X = np.stack([X_mel, X_cqt], axis=-1)  # (batch, 96, 259, 2)

# 🔍 Controllo forma finale
print("✅ Shape finale X:", X.shape)  # Deve dare (batch, 96, 259, 2)
print("✅ Shape finale y:", y.shape)

# 🔵 Split in train/val
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("✅ Train set:", X_train.shape, "Validation set:", X_val.shape)

# 🔵 Definizione CNN 2D
def create_cnn_model(input_shape, num_labels):
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D((2,2)),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_labels, activation='sigmoid')  # multilabel output
    ])
    return model

# 🔵 Crea e compila modello
input_shape = X_train.shape[1:]  # (96, 259, 2)
num_labels = y_train.shape[1]

model = create_cnn_model(input_shape, num_labels)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
               loss='binary_crossentropy',
               metrics=['binary_accuracy'])

model.summary()


✅ Shape finale X: (9154, 96, 259, 2)
✅ Shape finale y: (9154, 18)
✅ Train set: (7323, 96, 259, 2) Validation set: (1831, 96, 259, 2)
Model: "sequential_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_18 (Conv2D)          (None, 94, 257, 32)       608       
                                                                 
 max_pooling2d_12 (MaxPoolin  (None, 47, 128, 32)      0         
 g2D)                                                            
                                                                 
 conv2d_19 (Conv2D)          (None, 45, 126, 64)       18496     
                                                                 
 max_pooling2d_13 (MaxPoolin  (None, 22, 63, 64)       0         
 g2D)                                                            
                                                                 
 conv2d_20 (Conv2D)          (None, 20, 61, 128)     

In [19]:
# Callbacks for training
csv_logger = CSVLogger('training_model_mix.log', append=False)

early_stopping = EarlyStopping(monitor='val_loss', patience=100)

model_name = 'best_model_mix.h5'
model_checkpoint = ModelCheckpoint(model_name, monitor='val_binary_accuracy', save_best_only=True, verbose=1)

callbacks = [csv_logger, early_stopping, model_checkpoint]


# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=1000,
    batch_size=32,
    callbacks=callbacks
)

Epoch 1/1000
135/229 [================>.............] - ETA: 35s - loss: 0.5795 - binary_accuracy: 0.7671

KeyboardInterrupt: 

In [ ]:
# History plot
log_df = pd.read_csv('training_model_mix.log')
best_val_acc = max(log_df['val_binary_accuracy'])
best_val_acc_epoch = log_df['epoch'][log_df['val_binary_accuracy'] == best_val_acc].values[0]

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(log_df['epoch'], log_df['binary_accuracy'], label='Train')
plt.plot(log_df['epoch'], log_df['val_binary_accuracy'], label='Validation')
plt.scatter(best_val_acc_epoch, best_val_acc, color='red', label='Best Val Accuracy')
plt.title('Binary Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Binary Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(log_df['epoch'], log_df['loss'], label='Train')
plt.plot(log_df['epoch'], log_df['val_loss'], label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for validation and test sets
model.load_weights('best_model_mix.h5')

y_val_pred = model.predict(X_val).argmax(axis=1)
y_test_pred = model.predict(X_test).argmax(axis=1)

val_cm = confusion_matrix(y_val, y_val_pred)
test_cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
ConfusionMatrixDisplay(val_cm, display_labels=np.unique(y_train)).plot(cmap='Blues', ax=plt.gca())
plt.title("Validation Set Confusion Matrix")

plt.subplot(1, 2, 2)
ConfusionMatrixDisplay(test_cm, display_labels=np.unique(y_train)).plot(cmap='Blues', ax=plt.gca())
plt.title("Test Set Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Classification report for validation and test sets
print(classification_report(y_val, y_val_pred))
print(classification_report(y_test, y_test_pred))